In [2]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

llm = init_chat_model(
    model_name="qwen/qwen3.8-27b",
)
llm


In [12]:
from typing_extensions import TypedDict
from typing  import Annotated
from langgraph.graph import StateGraph,START , END
from langgraph.graph.message import add_messages
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import tools_condition , ToolNode
from langgraph.types import Command , interrupt
from langchain_core.tools import tool



class State(TypedDict):
    messages: Annotated[list,add_messages]
    
graph_builder = StateGraph(State)

@tool
def human_assistance(query:str)->str:
    """Request assistance from human"""
    human_response = interrupt({"query":query})
    return human_response["data"]
tool = TavilySearch(max_results=2)
tools = [tool,human_assistance]
llm_with_tool = llm.bind_tools(tools)

def chat_bot(state:State):
    message = llm_with_tool.invoke(state["messages"])
    return {"messages":[message]}
##nodes

graph_builder.add_node("chat_bot",chat_bot)
tool_node = ToolNode(tools)
graph_builder.add_node("tools",tool_node)

##edges
graph_builder.add_conditional_edges(
    "chat_bot",tools_condition
)
graph_builder.add_edge(
    "tools","chat_bot"
)
graph_builder.add_edge(START,"chat_bot")
graph_builder.add_edge("chat_bot",END)

checkpoint = MemorySaver()
graph = graph_builder.compile(checkpointer=checkpoint)

